# TechQA Knowledge Base — Section-aware BGE-M3 + Qdrant Cloud

Phiên bản này được điều chỉnh để phù hợp với cấu trúc thật của TechQA/APAR.

### Pipeline

`JSON → id + title + sections → light cleaning → section-aware chunking → context (APAR ID + title + section) → BGE-M3 Dense 1024 → deterministic point ID → Qdrant Cloud`

### Các thay đổi chính

- Đọc TechQA JSON bằng **streaming** với `ijson`.
- Không giữ toàn bộ corpus trong RAM.
- Tận dụng `sections` có sẵn trong TechQA thay vì chunk mù toàn bộ `text`.
- Cleaning nhẹ: chuẩn hóa line ending, whitespace và blank lines; **không xóa error code, version, APAR ID, command, URL hay metadata kỹ thuật**.
- Mỗi chunk được thêm context `APAR ID`, `Title`, `Section` trước khi embedding.
- Lưu `title`, `section`, `section_index`, `chunk_index` vào payload/metadata của Qdrant.
- Dùng **deterministic UUID5 point ID**, nên nếu Colab bị ngắt sau khi upload nhưng trước checkpoint thì chạy lại vẫn `upsert` cùng point, không tạo duplicate point.
- Checkpoint/resume theo document.
- Chỉ dùng **Dense BGE-M3 1024-dim**; không dùng Sparse/ColBERT.
- Pipeline tạo **collection mới** để không trộn với baseline cũ.

> Khuyến nghị: dùng Google Colab GPU T4 và lưu Qdrant credentials trong Colab Secrets.


In [1]:
# 1. Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Cấu hình dataset

Đặt `TechQA.tar.gz` trên Google Drive và sửa `TAR_PATH` nếu cần.

In [2]:
import os
import tarfile
import glob

TAR_PATH = "/content/drive/MyDrive/TechQA.tar.gz"
EXTRACT_PATH = "/content/TechQA_data"

os.makedirs(EXTRACT_PATH, exist_ok=True)

# Chỉ giải nén nếu chưa tìm thấy corpus
existing = glob.glob(
    f"{EXTRACT_PATH}/**/full_technote_collection.sections.json",
    recursive=True
)

if existing:
    print("Đã tìm thấy TechQA corpus, bỏ qua bước giải nén.")
    print(existing[0])
else:
    print(f"Đang giải nén: {TAR_PATH}")
    with tarfile.open(TAR_PATH, "r:gz") as tar:
        tar.extractall(path=EXTRACT_PATH)
    print("Giải nén hoàn tất.")

technotes_files = glob.glob(
    f"{EXTRACT_PATH}/**/full_technote_collection.sections.json",
    recursive=True
)

if not technotes_files:
    raise FileNotFoundError(
        "Không tìm thấy full_technote_collection.sections.json. "
        "Kiểm tra lại TechQA.tar.gz."
    )

TECHNOTES_FILE = technotes_files[0]
print("TECHNOTES_FILE =", TECHNOTES_FILE)

Đang giải nén: /content/drive/MyDrive/TechQA.tar.gz


/tmp/ipykernel_902/2774258109.py:22: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=EXTRACT_PATH)


Giải nén hoàn tất.
TECHNOTES_FILE = /content/TechQA_data/TechQA/technote_corpus/full_technote_collection.sections.json


In [3]:
import os
import json
# Pipeline mới dùng checkpoint riêng, KHÔNG dùng checkpoint của pipeline cũ.
CHECKPOINT_PATH = "/content/drive/MyDrive/techqa_qdrant_section_clean_checkpoint.json"

if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
        print(json.load(f))
else:
    print("Chưa có checkpoint. Pipeline sẽ bắt đầu từ document 0.")


Chưa có checkpoint. Pipeline sẽ bắt đầu từ document 0.


## 3. Cài đặt thư viện

`faiss-cpu` không còn cần thiết cho pipeline Qdrant Cloud này.

In [4]:
!pip install -q -U     langchain     langchain-huggingface     langchain-qdrant     langchain-text-splitters     qdrant-client     sentence-transformers     ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 47.4 MB/s eta 0:00:00


## 4. Cấu hình Qdrant Cloud

### Tạo 2 secrets trong Google Colab

Vào **Colab → Secrets** và tạo:

- `QDRANT_URL`
- `QDRANT_API_KEY`

Không hard-code API key trực tiếp vào notebook.

Ví dụ `QDRANT_URL` có dạng:

`https://<cluster-id>.<region>.qdrant.io`

Sau khi tạo secrets, cell dưới sẽ tự lấy credentials.

In [12]:
import os
from google.colab import userdata
QDRANT_URL = userdata.get('QDRANT_URL')
QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')

if not QDRANT_URL or not QDRANT_API_KEY:
    raise ValueError(
        "Chưa cấu hình QDRANT_URL / QDRANT_API_KEY. "
        "Hãy thêm 2 secrets này vào Colab → Secrets."
    )

# Collection MỚI cho pipeline cleaning + section-aware.
# Không dùng collection baseline cũ để tránh trộn hai pipeline.
COLLECTION_NAME = "techqa_corpus_bge_m3_section_clean"

print("Qdrant URL:", QDRANT_URL)
print("Collection:", COLLECTION_NAME)
print("API key: [hidden]")


Qdrant URL: "https://f3636289-cb7b-42b1-ab07-b17b6ef9c217.sa-east-1-0.aws.cloud.qdrant.io"
Collection: techqa_corpus_bge_m3_section_clean
API key: [hidden]


## 5. Các tham số ingestion

Các tham số này có thể tăng/giảm tùy RAM và VRAM.

Với Colab T4, nên bắt đầu:

- `DOC_BATCH_SIZE = 1000`
- `EMBED_BATCH_SIZE = 8`

Nếu ổn định, có thể tăng dần.

In [7]:
# ===== PIPELINE CONFIG =====

MAX_DOCS = None              # None = xử lý toàn bộ corpus
DOC_BATCH_SIZE = 500         # documents đọc mỗi batch
EMBED_BATCH_SIZE = 8         # chunks đưa vào BGE-M3 mỗi batch

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

# BGE-M3 dense output
VECTOR_SIZE = 1024

# Checkpoint riêng cho pipeline mới
CHECKPOINT_PATH = "/content/drive/MyDrive/techqa_qdrant_section_clean_checkpoint.json"

# False = tiếp tục theo checkpoint hiện tại.
# True = XÓA collection mới + checkpoint và chạy lại từ đầu.
RESET_COLLECTION = False

# Nếu muốn test trước:
# MAX_DOCS = 100

print("MAX_DOCS =", MAX_DOCS)
print("DOC_BATCH_SIZE =", DOC_BATCH_SIZE)
print("EMBED_BATCH_SIZE =", EMBED_BATCH_SIZE)
print("CHUNK_SIZE =", CHUNK_SIZE)
print("CHUNK_OVERLAP =", CHUNK_OVERLAP)
print("VECTOR_SIZE =", VECTOR_SIZE)
print("COLLECTION_NAME =", COLLECTION_NAME)


MAX_DOCS = None
DOC_BATCH_SIZE = 500
EMBED_BATCH_SIZE = 8
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
VECTOR_SIZE = 1024
COLLECTION_NAME = techqa_corpus_bge_m3_section_clean


## 6. Streaming reader

Không tạo `documents = [...]` cho toàn bộ dataset.

Generator dưới đây chỉ đưa một batch documents vào RAM tại một thời điểm.

In [8]:
import re
import ijson
from langchain_core.documents import Document

def clean_text(text: str) -> str:
    """Light cleaning: normalize formatting without removing technical information."""
    if text is None:
        return ""

    text = str(text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Strip trailing/leading whitespace on every line.
    lines = [line.strip() for line in text.split("\n")]

    # Collapse consecutive blank lines to at most one blank line.
    cleaned_lines = []
    previous_blank = False

    for line in lines:
        if not line:
            if not previous_blank:
                cleaned_lines.append("")
            previous_blank = True
        else:
            cleaned_lines.append(line)
            previous_blank = False

    text = "\n".join(cleaned_lines)

    # Normalize repeated spaces/tabs inside a line.
    text = re.sub(r"[ \t]+", " ", text)

    # Keep paragraph boundaries, but remove excessive blank lines.
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def build_section_documents(doc_id, item):
    """
    Convert one TechQA record into section-aware LangChain Documents.

    Logic:
    1. Nếu có ít nhất một section có nội dung -> dùng các section đó.
    2. Nếu sections không tồn tại, rỗng, hoặc tất cả section đều rỗng
       -> fallback về full document text.
    """

    if not isinstance(item, dict):
        item = {"content": item}

    # --------------------------------------------------
    # 1. Basic information
    # --------------------------------------------------
    doc_id = str(item.get("id") or doc_id)
    title = clean_text(item.get("title") or "")

    sections = item.get("sections")

    # --------------------------------------------------
    # 2. Kiểm tra sections có thực sự usable hay không
    # --------------------------------------------------
    valid_sections = []

    if isinstance(sections, list):

        for section_index, section in enumerate(sections):

            if not isinstance(section, dict):
                continue

            section_name = clean_text(
                section.get("id")
                or f"SECTION_{section_index}"
            )

            section_text = clean_text(
                section.get("text") or ""
            )

            # Chỉ giữ section có nội dung thật
            if section_text:
                valid_sections.append(
                    {
                        "section_index": section_index,
                        "section": section_name,
                        "text": section_text,
                    }
                )

    # --------------------------------------------------
    # 3. Nếu có section usable -> dùng sections
    # --------------------------------------------------
    if valid_sections:

        for section in valid_sections:

            yield Document(
                page_content=section["text"],
                metadata={
                    "id": doc_id,
                    "source_id": doc_id,
                    "title": title,
                    "section": section["section"],
                    "section_index": section["section_index"],
                }
            )

        return

    # --------------------------------------------------
    # 4. Không có section usable -> fallback full text
    # --------------------------------------------------
    full_text = clean_text(
        item.get("text")
        or item.get("document")
        or item.get("content")
        or ""
    )

    if full_text:

        yield Document(
            page_content=full_text,
            metadata={
                "id": doc_id,
                "source_id": doc_id,
                "title": title,
                "section": "FULL_DOCUMENT",
                "section_index": 0,
            }
        )


def iter_techqa_documents(file_path, start_doc=0, max_docs=None):
    """Stream TechQA records without loading the whole JSON into RAM."""
    processed = 0
    yielded = 0

    with open(file_path, "rb") as f:
        try:
            objects = ijson.kvitems(f, "")

            for doc_id, item in objects:
                if processed < start_doc:
                    processed += 1
                    continue

                if max_docs is not None and yielded >= max_docs:
                    return

                docs_for_record = list(build_section_documents(doc_id, item))

                # One source APAR counts as one processed document.
                for doc in docs_for_record:
                    yield doc

                yielded += 1

        except Exception as e:
            print("Dictionary streaming failed; trying list streaming.")
            print("Reason:", e)

            f.seek(0)
            objects = ijson.items(f, "item")

            for idx, item in enumerate(objects):
                if processed < start_doc:
                    processed += 1
                    continue

                if max_docs is not None and yielded >= max_docs:
                    return

                if not isinstance(item, dict):
                    continue

                doc_id = item.get("id", f"doc_{start_doc + yielded}")

                for doc in build_section_documents(doc_id, item):
                    yield doc

                yielded += 1


def document_batches(file_path, batch_size, start_doc=0, max_docs=None):
    """Yield source APAR records as batches while preserving section documents."""
    batch = []
    source_docs_in_batch = 0

    # We need source-document boundaries for checkpointing.
    # Use a generator that yields (source_id, section_docs).
    def iter_source_records():
        processed = 0
        yielded = 0

        with open(file_path, "rb") as f:
            try:
                objects = ijson.kvitems(f, "")

                for doc_id, item in objects:
                    if processed < start_doc:
                        processed += 1
                        continue
                    if max_docs is not None and yielded >= max_docs:
                        return

                    docs_for_record = list(build_section_documents(doc_id, item))
                    yield str(item.get("id") if isinstance(item, dict) and item.get("id") else doc_id), docs_for_record
                    yielded += 1

            except Exception as e:
                print("Dictionary streaming failed; trying list streaming.")
                print("Reason:", e)

                f.seek(0)
                objects = ijson.items(f, "item")

                for idx, item in enumerate(objects):
                    if processed < start_doc:
                        processed += 1
                        continue
                    if max_docs is not None and yielded >= max_docs:
                        return
                    if not isinstance(item, dict):
                        continue

                    doc_id = item.get("id", f"doc_{start_doc + yielded}")
                    docs_for_record = list(build_section_documents(doc_id, item))
                    yield str(doc_id), docs_for_record
                    yielded += 1

    for source_id, section_docs in iter_source_records():
        batch.append((source_id, section_docs))
        source_docs_in_batch += 1

        if source_docs_in_batch >= batch_size:
            yield batch
            batch = []
            source_docs_in_batch = 0

    if batch:
        yield batch


## 7. Checkpoint utilities

Checkpoint lưu số documents đã xử lý thành công.

Nếu Colab disconnect sau batch 120, notebook có thể tiếp tục từ batch tiếp theo thay vì embedding lại từ đầu.

In [9]:
import os
import json

def load_checkpoint(path):
    if not os.path.exists(path):
        return {
            "processed_docs": 0,
            "processed_chunks": 0,
        }

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_checkpoint(path, processed_docs, processed_chunks):
    payload = {
        "processed_docs": int(processed_docs),
        "processed_chunks": int(processed_chunks),
    }

    tmp_path = path + ".tmp"

    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

    os.replace(tmp_path, path)


checkpoint = load_checkpoint(CHECKPOINT_PATH)

if RESET_COLLECTION:
    checkpoint = {
        "processed_docs": 0,
        "processed_chunks": 0,
    }

print("Checkpoint:", checkpoint)


Checkpoint: {'processed_docs': 0, 'processed_chunks': 0}


## 8. Khởi tạo BGE-M3 + Qdrant Cloud

BGE-M3 được dùng thông qua `HuggingFaceEmbeddings`.

Qdrant Cloud được kết nối bằng `QdrantClient(url=..., api_key=...)`.

In [13]:
import gc
import hashlib
import uuid
import torch

from langchain_huggingface import HuggingFaceEmbeddings
from qdrant_client import QdrantClient, models

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Embedding device:", device.upper())

if device == "cpu":
    print("WARNING: GPU chưa được bật. BGE-M3 trên CPU sẽ rất chậm.")

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": device},
    encode_kwargs={
        "batch_size": EMBED_BATCH_SIZE,
        "normalize_embeddings": True,
    },
)

client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=120,
)

print("Connected to Qdrant Cloud.")

existing_collection = False

try:
    client.get_collection(COLLECTION_NAME)
    existing_collection = True
    print(f"Collection '{COLLECTION_NAME}' đã tồn tại.")
except Exception:
    print(f"Collection '{COLLECTION_NAME}' chưa tồn tại.")

if RESET_COLLECTION and existing_collection:
    print("RESET_COLLECTION=True -> xóa collection pipeline mới...")
    client.delete_collection(COLLECTION_NAME)
    existing_collection = False
    print("Collection đã được xóa.")

if not existing_collection:
    print("Creating Qdrant collection...")
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=VECTOR_SIZE,
            distance=models.Distance.COSINE,
        ),
    )
    existing_collection = True
    print("Collection created.")

# Deterministic namespace: changing it would create different point IDs.
POINT_NAMESPACE = uuid.uuid5(
    uuid.NAMESPACE_URL,
    "techqa-bge-m3-section-clean-v1"
)

def deterministic_point_id(source_id, section_index, chunk_index):
    """Stable UUID for one logical TechQA chunk.

    Same source/section/chunk always maps to the same Qdrant point ID.
    This makes retrying an already-uploaded batch safe via upsert.
    """
    key = f"{source_id}|section={section_index}|chunk={chunk_index}"
    return str(uuid.uuid5(POINT_NAMESPACE, key))

print("Deterministic point IDs: enabled")


Embedding device: CUDA


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Connected to Qdrant Cloud.
Collection 'techqa_corpus_bge_m3_section_clean' chưa tồn tại.
Creating Qdrant collection...


ResponseHandlingException: [Errno -2] Name or service not known

## 9. Batch chunking + embedding + Qdrant Cloud upload

Đây là phần quan trọng nhất.

Pipeline:

`documents → chunks → embeddings → Qdrant Cloud`

chỉ xử lý một batch tại một thời điểm.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm.auto import tqdm

# Section-aware chunking:
# each section is chunked independently, so chunks do not cross semantic
# boundaries such as ERROR DESCRIPTION -> LOCAL FIX -> PROBLEM CONCLUSION.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", " ", ""],
)

processed_docs = checkpoint["processed_docs"]
processed_chunks = checkpoint["processed_chunks"]

remaining_docs = None
if MAX_DOCS is not None:
    remaining_docs = max(0, MAX_DOCS - processed_docs)

print("Starting from document:", processed_docs)
print("Previously uploaded chunks:", processed_chunks)

batch_number = 0

for source_batch in document_batches(
    TECHNOTES_FILE,
    batch_size=DOC_BATCH_SIZE,
    start_doc=processed_docs,
    max_docs=remaining_docs,
):
    batch_number += 1

    print(f"\n===== Batch {batch_number} =====")
    print(f"Source documents in batch: {len(source_batch)}")

    # Chunk each section independently.
    all_chunks = []
    source_doc_count = 0

    for source_id, section_docs in source_batch:
        source_doc_count += 1

        for section_doc in section_docs:
            section_index = int(section_doc.metadata.get("section_index", 0))

            section_chunks = text_splitter.split_documents([section_doc])

            for chunk_index, chunk in enumerate(section_chunks):
                chunk.metadata["chunk_index"] = chunk_index

                # Store the source document id in a dedicated field as well.
                chunk.metadata["source_id"] = source_id

                # Prepend context AFTER splitting so EVERY chunk has the context
                header = (
                    f"APAR ID: {source_id}\n"
                    f"Title: {chunk.metadata.get('title', '')}\n"
                    f"Section: {chunk.metadata.get('section', '')}\n\n"
                )
                chunk.page_content = header + chunk.page_content

                all_chunks.append(chunk)

    chunks = all_chunks
    print(f"Chunks in batch: {len(chunks)}")

    if chunks:
        # Embed ourselves so point IDs and payload are fully deterministic.
        texts = [doc.page_content for doc in chunks]
        vectors = embeddings.embed_documents(texts)

        points = []

        for doc, vector in zip(chunks, vectors):
            source_id = str(doc.metadata["source_id"])
            section_index = int(doc.metadata.get("section_index", 0))
            chunk_index = int(doc.metadata.get("chunk_index", 0))

            point_id = deterministic_point_id(
                source_id,
                section_index,
                chunk_index,
            )

            payload = {
                "page_content": doc.page_content,
                "metadata": {
                    "id": source_id,
                    "source_id": source_id,
                    "title": doc.metadata.get("title", ""),
                    "section": doc.metadata.get("section", ""),
                    "section_index": section_index,
                    "chunk_index": chunk_index,
                },
            }

            points.append(
                models.PointStruct(
                    id=point_id,
                    vector=vector,
                    payload=payload,
                )
            )

        # Upsert is intentional:
        # retrying the same batch updates the same deterministic IDs.
        for start in range(0, len(points), EMBED_BATCH_SIZE):
            client.upsert(
                collection_name=COLLECTION_NAME,
                points=points[start:start + EMBED_BATCH_SIZE],
                wait=True,
            )

        processed_chunks += len(points)

    # Only advance source-document checkpoint after the entire batch
    # has been successfully upserted.
    processed_docs += source_doc_count

    save_checkpoint(
        CHECKPOINT_PATH,
        processed_docs,
        processed_chunks,
    )

    print(
        f"Uploaded successfully | "
        f"docs={processed_docs:,} | "
        f"chunks={processed_chunks:,}"
    )

    del source_batch
    del chunks

    if "vectors" in locals():
        del vectors
    if "points" in locals():
        del points

    gc.collect()

    if device == "cuda":
        torch.cuda.empty_cache()

print("\n===== INGESTION COMPLETED =====")
print("Processed documents:", processed_docs)
print("Processed chunks:", processed_chunks)
print("Checkpoint:", CHECKPOINT_PATH)


## 10. Kiểm tra Qdrant Cloud

Kiểm tra collection và số lượng vectors/points đã upload.

In [ ]:
info = client.get_collection(COLLECTION_NAME)

print("Collection:", COLLECTION_NAME)
print("Points count:", info.points_count)
print("Status:", info.status)
print("Vector config:", info.config.params.vectors)

## 11. Load retriever từ Qdrant Cloud

Không cần tải database về Google Drive.

Notebook kết nối trực tiếp đến Qdrant Cloud.

In [ ]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings

device = "cuda" if torch.cuda.is_available() else "cpu"

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": device},
    encode_kwargs={
        "batch_size": EMBED_BATCH_SIZE,
        "normalize_embeddings": True,
    },
)

print("Connected to Qdrant Cloud collection:", COLLECTION_NAME)
print("Dense vector size:", VECTOR_SIZE)

# Use the native Qdrant client for retrieval so the payload structure
# remains exactly the same as the ingestion pipeline.


## 12. Test semantic search

In [ ]:
from qdrant_client import models

query = "How to configure WebSphere Application Server?"

print(f"🔍 Query: {query}\n")

query_vector = embeddings.embed_query(query)

search_result = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=5,
    with_payload=True,
)

for i, point in enumerate(search_result.points, 1):
    payload = point.payload or {}
    metadata = payload.get("metadata", {})

    print(f"--- Result {i} ---")
    print("Score:", point.score)
    print("Document ID:", metadata.get("id", "N/A"))
    print("Title:", metadata.get("title", "N/A"))
    print("Section:", metadata.get("section", "N/A"))
    print("Section index:", metadata.get("section_index", "N/A"))
    print("Chunk index:", metadata.get("chunk_index", "N/A"))
    print("Content:", payload.get("page_content", "")[:700].replace("\n", " "))
    print()


## 13. Test search với nhiều câu hỏi

Có thể thay danh sách dưới đây bằng các câu hỏi trong validation set của TechQA.

In [ ]:
from qdrant_client import models

test_queries = [
    "How to configure WebSphere Application Server?",
    "How do I install Web GUI?",
    "What is required before installing WebSphere?",
    "IllegalMonitorStateException JIT compiled code",
    "What is the local fix for the APAR?",
]

for query in test_queries:
    print("=" * 90)
    print("QUERY:", query)

    query_vector = embeddings.embed_query(query)

    search_result = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=5,
        with_payload=True,
    )

    for rank, point in enumerate(search_result.points, 1):
        payload = point.payload or {}
        metadata = payload.get("metadata", {})

        print(
            f"{rank}. score={point.score:.4f} | "
            f"id={metadata.get('id')} | "
            f"section={metadata.get('section')}"
        )

        print(
            "   ",
            payload.get("page_content", "")[:250]
            .replace("\n", " ")
        )

    print()


# Ghi chú vận hành

### 1. Pipeline mới dùng collection riêng

Collection:

```text
techqa_corpus_bge_m3_section_clean
```

Collection cũ:

```text
techqa_corpus_bge_m3
```

được giữ làm **baseline** và không bị trộn dữ liệu.

### 2. Checkpoint

Checkpoint mới:

```text
/content/drive/MyDrive/techqa_qdrant_section_clean_checkpoint.json
```

Nếu Colab disconnect, chạy lại notebook. Các document trước checkpoint sẽ được bỏ qua.

### 3. Deterministic point ID

Point ID được tạo từ:

```text
APAR ID + section_index + chunk_index
```

bằng UUID5.

Vì vậy nếu batch cuối cùng đã upload lên Qdrant nhưng Colab chết trước khi checkpoint được ghi, chạy lại batch đó sẽ `upsert` vào **cùng point ID**, không tạo point duplicate.

### 4. Cleaning

Cleaning chỉ chuẩn hóa:

- `CRLF/CR → LF`
- whitespace dư
- blank lines dư
- indentation không cần thiết

Không xóa:

- APAR ID
- error code
- version/release
- component ID
- command
- exception name
- URL
- các field kỹ thuật khác

### 5. Section-aware chunking

Nếu document có `sections`, mỗi section được chunk riêng.

Ví dụ:

```text
ERROR DESCRIPTION
    ↓
chunks

LOCAL FIX
    ↓
chunks

PROBLEM CONCLUSION
    ↓
chunks
```

Một chunk không tự động vượt từ section này sang section khác.

### 6. Nếu muốn test trước

Đặt:

```python
MAX_DOCS = 100
```

Sau khi kiểm tra Qdrant và search ổn, đổi:

```python
MAX_DOCS = None
```

### 7. Nếu muốn chạy lại toàn bộ pipeline mới

Đặt:

```python
RESET_COLLECTION = True
```

rồi chạy lại từ phần cấu hình/ingestion.

> Không dùng checkpoint cũ `techqa_qdrant_checkpoint.json` cho pipeline này vì cách chunking và point ID đã thay đổi.

### 8. Security

Không hard-code `QDRANT_API_KEY` vào notebook. Nếu API key cũ đã từng được lưu trong notebook hoặc chia sẻ, nên rotate/revoke key đó và tạo key mới trong Qdrant Cloud.
